# Sim-to-Real Translator (Local)

Trains and evaluates a **hierarchical "translator" policy** on top of the single-agent hover policy from `MARL_Crazyflie.ipynb`.

A low-level PPO policy is trained once, then frozen. Porting it to different physics (a different simulator, or a real drone) rarely works 1:1 — e.g. a policy trained where hovering takes a commanded thrust of ~0.3 might find the same command launches a "real" drone into the ceiling, because the real actuators are stronger per unit command. `cf_translator_env.py` wraps the frozen policy with a small high-level "translator" policy that watches recent drone history (positions, orientations, the low-level policy's raw actions) and outputs a per-channel scale `[thrust_scale, roll_scale, pitch_scale, yaw_scale]` applied to the low-level action before it reaches the (deliberately mismatched) physics. Physics are randomized heavily per episode — actuator gain mismatch, mass/inertia/actuator noise, a wind-like force, an optional gravity tilt — so the translator has to notice something's off and correct for it.

This is trained and evaluated **sim-to-sim** (the "real world" is just a more heavily randomized simulation) — a sanity check before ever attempting an actual sim-to-real port.

**Requires the same local venv as `MARL_Crazyflie.ipynb`.** Follow that notebook's Setup section first if you haven't, then select the same kernel here.

New module files this notebook uses (alongside everything from `MARL_Crazyflie.ipynb`):

| Module | What it holds |
|---|---|
| `cf_translator_env.py` | `CrazyflieTranslatorEnv` — the hierarchical translator environment |
| `translator_eval.py` | Batched, apples-to-apples comparison between "no correction" and "with translator" |


## Setup


In [1]:
import os
import sys
import pathlib

NOTEBOOK_DIR = pathlib.Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import jax
import mujoco

print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("MuJoCo version:", mujoco.__version__)


JAX backend: gpu
JAX devices: [CudaDevice(id=0)]
MuJoCo version: 3.4.0


## Low-Level Policy: Single-Agent PPO Hover

The policy the translator will learn to correct for. Set `LOAD_EXISTING_LOW_LEVEL_POLICY = True` below to load a previously-trained checkpoint (default: `constants.PPO_MODEL_SAVE_PATH`) instead of training from scratch, or `False` to train the same single-drone PPO hover policy as `MARL_Crazyflie.ipynb`'s "PPO Training" section.


In [2]:
from mujoco_playground import wrapper

from constants import SCENE_PATH_1_DRONE, PPO_MODEL_SAVE_PATH
from crazyflie_env import CrazyflieEnv
from ppo_training import default_ppo_params, build_ppo_train_fn, load_ppo_policy
from training_plots import PPOProgressPlotter

low_level_env = CrazyflieEnv(scene_path=SCENE_PATH_1_DRONE, num_drones=1)

LOAD_EXISTING_LOW_LEVEL_POLICY = True  # False to train a fresh low-level policy instead

if LOAD_EXISTING_LOW_LEVEL_POLICY:
    make_inference_fn, low_level_params = load_ppo_policy(PPO_MODEL_SAVE_PATH, low_level_env)
    print(f"Loaded low-level policy from {PPO_MODEL_SAVE_PATH}")
else:
    ppo_params = default_ppo_params(num_timesteps=60_000_000)
    progress = PPOProgressPlotter(num_timesteps=ppo_params["num_timesteps"])
    train_fn = build_ppo_train_fn(ppo_params, progress_fn=progress)

    make_inference_fn, low_level_params, metrics = train_fn(
        environment=low_level_env,
        wrap_env_fn=wrapper.wrap_for_brax_training,
    )
    progress.print_timing()


Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
Loaded low-level policy from /mnt/c/Users/xboxa/Coding/multi-agentCrazyflieLift/code/src/gpu_training/../../rl_models/gpu/PPO_brax_model_checkpoint.pkl


### Wrap the low-level policy

`CrazyflieTranslatorEnv` expects a frozen, deterministic, Brax-style `(obs, rng) -> (action, extras)` function — exactly what `make_inference_fn(params, deterministic=True)` gives us. It is never updated while training the translator.


In [3]:
low_level_inference_fn = make_inference_fn(low_level_params, deterministic=True)


## Translator Environment & Training

`CrazyflieTranslatorEnv` is a drop-in `environment=` for the same `ppo_training.py` / Brax PPO pipeline used above, just with a much smaller action/observation space — likely needs far fewer timesteps than the low-level policy. Set `LOAD_EXISTING_TRANSLATOR_POLICY = True` below to load a previously-saved translator checkpoint (default: `constants.TRANSLATOR_MODEL_SAVE_PATH`, written by the "Save Translator Model" cell) instead of training from scratch. Defaults below are a starting point, not a tuned recipe; watch the reward curve and adjust `num_timesteps`/`num_envs` as needed.


In [4]:
from cf_translator_env import CrazyflieTranslatorEnv, TranslatorDomainRandomization
from constants import TRANSLATOR_MODEL_SAVE_PATH
from ppo_training import load_ppo_policy

translator_env = CrazyflieTranslatorEnv(
    low_level_inference_fn=low_level_inference_fn,
    scene_path=SCENE_PATH_1_DRONE,
    history_len=4,
    scale_range=(0.5, 2.0),  # halving to doubling -- covers the sampled thrust/torque gap ranges
    randomization=TranslatorDomainRandomization(),  # defaults -- tune here for harder/easier gaps
    randomize_gravity=True,
)

LOAD_EXISTING_TRANSLATOR_POLICY = True  # True to load a previously-saved translator instead of training

if LOAD_EXISTING_TRANSLATOR_POLICY:
    translator_make_inference_fn, translator_params = load_ppo_policy(TRANSLATOR_MODEL_SAVE_PATH, translator_env)
    print(f"Loaded translator policy from {TRANSLATOR_MODEL_SAVE_PATH}")
else:
    translator_ppo_params = default_ppo_params(
        num_timesteps=10_000_000,
        num_envs=512,
        episode_length=1500,
        # Brax's actual per-eval-epoch step count is batch_size * unroll_length *
        # num_minibatches (independent of num_envs) -- the stock num_minibatches=32
        # rounds a 10M-step run up to ~18M actual steps. Lower it here so the
        # rounding overshoot stays small relative to num_timesteps.
        num_minibatches=8,
    )
    translator_progress = PPOProgressPlotter(num_timesteps=translator_ppo_params["num_timesteps"])
    translator_train_fn = build_ppo_train_fn(translator_ppo_params, progress_fn=translator_progress)

    translator_make_inference_fn, translator_params, translator_metrics = translator_train_fn(
        environment=translator_env,
        wrap_env_fn=wrapper.wrap_for_brax_training,
    )
    translator_progress.print_timing()


Loaded translator policy from /mnt/c/Users/xboxa/Coding/multi-agentCrazyflieLift/code/src/gpu_training/../../rl_models/gpu/translator_brax_model_checkpoint.pkl


### Nominal-physics environment (sanity check)

A second `CrazyflieTranslatorEnv`, same wrapped low-level policy, but with `TranslatorDomainRandomization.nominal()` (gap gains pinned to 1.0, all extra noise/wind/gravity-tilt off) -- i.e. the same conditions the low-level policy was actually trained under. Appended to each rollout below: if the wrapped policy struggles even here, that points to a bug in the wrapping rather than the domain randomization being hard.


In [5]:
nominal_translator_env = CrazyflieTranslatorEnv(
    low_level_inference_fn=low_level_inference_fn,
    scene_path=SCENE_PATH_1_DRONE,
    history_len=4,
    scale_range=(0.5, 2.0),
    randomization=TranslatorDomainRandomization.nominal(),
    randomize_gravity=False,
)


## Translator Rollout (visual sanity check)

Watch the translator fly under one randomly-sampled sim-to-real gap.


In [6]:
from rollout_utils import rollout_policy

translator_inference_fn = translator_make_inference_fn(translator_params, deterministic=True)

COMPARISON_SEED = 7

rollout = rollout_policy(
    translator_env,
    params=None,
    apply_fn=translator_inference_fn,
    episode_length=1500,
    seed=COMPARISON_SEED,
    print_logs=False,
    random_reset=True,
    use_brax_policy=True,
)


Target: [-1.0433593  -0.44845533  0.5544043 ]
Observation shape for 1 drones: (64,)

Total steps: 76, Final reward: 0.42

Rendering trajectory, target at [-1.0433593  -0.44845533  0.5544043 ]...


100%|██████████| 76/76 [00:02<00:00, 26.27it/s]


This browser does not support the video tag.


In [7]:
print("Nominal physics (no extra randomization):")
nominal_rollout = rollout_policy(
    nominal_translator_env,
    params=None,
    apply_fn=translator_inference_fn,
    episode_length=1500,
    seed=COMPARISON_SEED,
    print_logs=False,
    random_reset=True,
    use_brax_policy=True,
)


Nominal physics (no extra randomization):
Target: [-1.0433593  -0.44845533  0.5544043 ]
Observation shape for 1 drones: (64,)

Total steps: 37, Final reward: -0.86

Rendering trajectory, target at [-1.0433593  -0.44845533  0.5544043 ]...


100%|██████████| 37/37 [00:01<00:00, 23.31it/s]


This browser does not support the video tag.


## Baseline Rollout (no correction)

The same low-level policy, same seed (so the same sampled sim-to-real gap), but with the translator's scale fixed at `[1, 1, 1, 1]` — i.e. running the low-level policy unmodified. Compare against the rollout above.


In [8]:
from translator_eval import identity_scale_policy

baseline_rollout = rollout_policy(
    translator_env,
    params=None,
    apply_fn=identity_scale_policy,
    episode_length=1500,
    seed=COMPARISON_SEED,
    print_logs=False,
    random_reset=True,
    use_brax_policy=True,
)


Target: [-1.0433593  -0.44845533  0.5544043 ]
Observation shape for 1 drones: (64,)

Total steps: 77, Final reward: 0.36

Rendering trajectory, target at [-1.0433593  -0.44845533  0.5544043 ]...


100%|██████████| 77/77 [00:02<00:00, 26.48it/s]


This browser does not support the video tag.


In [ ]:
print("Nominal physics (no extra randomization):")
nominal_baseline_rollout = rollout_policy(
    nominal_translator_env,
    params=None,
    apply_fn=identity_scale_policy,
    episode_length=1500,
    seed=COMPARISON_SEED,
    print_logs=False,
    random_reset=True,
    use_brax_policy=True,
)


## Batched Comparison Across Many Randomized Physics Instances

A single rollout is anecdotal. This runs both policies across `NUM_SEEDS` independent, randomly-sampled physics instances (same seeds for both) and reports mean episode reward, mean steps survived, and the fraction of episodes that ran the full `EPISODE_LENGTH` without crashing/going out of bounds.


In [ ]:
from translator_eval import compare_policies, print_comparison_table

NUM_SEEDS = 64
EPISODE_LENGTH = 1500

results = compare_policies(
    translator_env,
    low_level_only_fn=identity_scale_policy,
    translator_fn=translator_inference_fn,
    num_seeds=NUM_SEEDS,
    episode_length=EPISODE_LENGTH,
    base_seed=0,
)

print_comparison_table(results, episode_length=EPISODE_LENGTH)


In [ ]:
nominal_results = compare_policies(
    nominal_translator_env,
    low_level_only_fn=identity_scale_policy,
    translator_fn=translator_inference_fn,
    num_seeds=NUM_SEEDS,
    episode_length=EPISODE_LENGTH,
    base_seed=0,
)
print("Nominal physics (no extra randomization):")
print_comparison_table(nominal_results, episode_length=EPISODE_LENGTH)


## Save Translator Model


In [ ]:
from brax.io import model

from constants import TRANSLATOR_MODEL_SAVE_PATH

os.makedirs(os.path.dirname(TRANSLATOR_MODEL_SAVE_PATH), exist_ok=True)
model.save_params(TRANSLATOR_MODEL_SAVE_PATH, translator_params)
print(f"Saved translator params to {TRANSLATOR_MODEL_SAVE_PATH}")
